<a href="https://colab.research.google.com/github/TonmoyTalukder/Bangla-Key2Text/blob/main/Bangla-KeywordExtractor/Extract_Keywords_from_Text_using_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
import requests
import pandas as pd
import random

class KeywordExtractor:
    def __init__(self):
        self.tokenizer = None
        self.model = None

    def load_model(self):
        model_name = 'csebuetnlp/banglabert'
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)

    def score_words(self, sentence):
        input_ids = torch.tensor([self.tokenizer.encode(sentence, max_length=512, padding='max_length', truncation=True)])
        # input_ids = torch.tensor([tokenizer.encode(sentence)])

        with torch.no_grad():
            outputs = self.model(input_ids=input_ids)
            embeddings = outputs.last_hidden_state.squeeze(0)

        mean_embedding = embeddings.mean(dim=0)
        word_scores = []
        for i in range(embeddings.size(0)):
            cos_sim = torch.nn.functional.cosine_similarity(embeddings[i], mean_embedding, dim=0)
            word_scores.append((self.tokenizer.decode(input_ids[0][i]), cos_sim.item()))

        return word_scores

    def clean(self, sen):
        strs = ''
        for i in range(len(sen)):
            if sen[i] != '।':
                strs = strs + sen[i]
        return strs.split()

    def print_top_values(self, data):
        data_shuffled = random.sample(data, len(data))
        top_values = int(len(data_shuffled) * 0.6)
        if top_values < 10:
            top_values = int(len(data_shuffled) * 0.7)
        if top_values < 4:
            top_values = int(len(data_shuffled) * 0.8)

        finalLst = []
        for i in range(top_values):
            finalLst.append(data_shuffled[i][0])

        return finalLst

    def keysOfSentence(self, sentence):
        lst = self.clean(sentence)
        lst2 = []
        final = []
        word_scores = self.score_words(sentence)

        for i in range(len(word_scores)):
            if i > 0:
                if word_scores[i][0] != '।':
                    lst2.append(word_scores[i][1])
        for i in range(len(lst)):
            final.append(tuple([lst[i], lst2[i]]))

        finalKeysLst = self.print_top_values(final)
        finalKeys = finalKeysLst

        return finalKeys

    def extract_keywords(self, text):
        self.load_model()
        return self.keysOfSentence(text)

In [ ]:
# Example usage
extractor = KeywordExtractor()
text = 'আমি বাংলায় গান শোনা ভালবাসি।'
keywords = extractor.extract_keywords(text)
print(keywords)

Some weights of the model checkpoint at csebuetnlp/banglabert were not used when initializing ElectraModel: ['discriminator_predictions.dense_prediction.weight', 'discriminator_predictions.dense_prediction.bias', 'discriminator_predictions.dense.bias', 'discriminator_predictions.dense.weight']
- This IS expected if you are initializing ElectraModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ElectraModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


['ভালবাসি', 'আমি', 'বাংলায়', 'গান']
